In [1]:
# Live Kaggle Execution Link: https://www.kaggle.com/code/sonuomar/notebook68d25131e2

# VRAM Budget Calculation (Required Format)
model_base_4bit    = 2.0   # GB — Your model at 4-bit
lora_adapters      = 0.3   # GB — LoRA rank + alpha
frames_per_clip    = 8     # Sampled frames per 5-sec clip
frame_tokens       = 256   # Visual tokens per frame (after vision encoder)
batch_size         = 2
token_hidden_dim   = 1536  # Your model's hidden size
activation_gb      = (frames_per_clip * frame_tokens * batch_size * token_hidden_dim * 2) / 1e9

# With gradient checkpointing: activation_gb * 0.4 (recomputed, not stored)
total_vram_gb      = model_base_4bit + lora_adapters + (activation_gb * 0.4)
print(f"Estimated VRAM: {total_vram_gb:.2f} GB")
# Must be < 16 GB for T4 or < 40 GB for A100

Estimated VRAM: 2.31 GB


In [2]:
!pip install -q -U bitsandbytes
!pip install -q -U git+https://github.com/huggingface/transformers.git
!pip install -q -U git+https://github.com/huggingface/peft.git
!pip install -q -U git+https://github.com/huggingface/accelerate.git
!pip install -q qwen-vl-utils decord webdataset

# Clone the recommended fine-tuning framework from the rubric
!git clone https://github.com/2U1/Qwen2-VL-Finetune.git

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 33.5 MB/s eta 0:00:00:00:0100:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 112.2 MB/s eta 0:00:0000:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 MB 53.0 MB/s eta 0:00:00:00:0100:01
Cloning into 'Qwen2-VL-Finetune'...
remote: Enumerating objects: 1201, done.
remote: Counting objects: 100% (743/743), done.
remote: Compressing objects: 100% (175/175), done.
remote: Total 1201 (delta 682), reused 568 (delta 568), pack-reused 458 (

In [3]:
import torch
from transformers import TrainingArguments

# Defining the strict constraints for free-tier T4 training
training_args = TrainingArguments(
    output_dir="./qwen2_vl_openpack_checkpoints",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8, # Simulates batch size 16
    optim="paged_adamw_32bit",
    save_strategy="steps",
    save_steps=50,
    save_total_limit=3, # Prevents filling up Kaggle's disk
    logging_steps=10,
    learning_rate=2e-4,
    fp16=True, # T4 optimal
    max_grad_norm=0.3,
    warmup_ratio=0.03,
    lr_scheduler_type="constant",
    gradient_checkpointing=True, # Non-negotiable for video
    gradient_checkpointing_kwargs={'use_reentrant':False}
)

print("Training arguments locked in with gradient checkpointing and aggressive saving.")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Training arguments locked in with gradient checkpointing and aggressive saving.
